# 🚌 Transit Delay Prediction

**Goal:** predict, *before a trip departs*, (a) whether it will arrive more than 5 minutes late and (b) by how many minutes, using timetable, weather and event data.

**How to run (Google Colab):** `Runtime → Run all`. The full run takes roughly 10–15 minutes on the free CPU runtime (most of it is hyperparameter tuning). The last cell downloads a zip containing the trained models, figures, results tables and `src/` code, ready to drop into the GitHub repo.

| Phase | Section |
|---|---|
| 1 | Problem framing & setup |
| 2 | Data understanding & cleaning |
| 3 | Exploratory data analysis |
| 4 | Feature engineering |
| 5 | Modeling (baselines → tuned boosting) |
| 6 | Explainability (SHAP) |
| 7 | Export artefacts for the Streamlit app |

---
# Phase 1 — Problem framing

**Business problem.** Unreliable service drives riders away faster than slow service does, because people plan around *variance*. A pre-departure delay forecast lets:
- **operations / dispatch** add capacity or adjust headways before a delay builds up,
- **rider apps** show "likely +8 min" and suggest alternatives,
- **planners** find structurally fragile routes that need timetable padding.

**Two tasks**

| | Regression | Classification |
|---|---|---|
| Target | `actual_arrival_delay_min` | `is_delayed` = arrival delay > 5 min |
| Primary metric | **MAE** (minutes, easy to explain, robust to extreme values) | **ROC-AUC** (threshold-free ranking quality) |
| Secondary metrics | RMSE (punishes big misses), R² | PR-AUC (honest under class imbalance), F1 at a tuned threshold |

**Why not accuracy?** If ~70% of trips are on time, predicting "on time" for every trip scores 70% and is useless.

**What counts as success:** a model has to clearly beat a *mean predictor* (regression) and *logistic regression* (classification). We also run a **permutation test**, which checks whether the model has learned real signal at all.

**The hard rule:** we may only use information available *before departure*. `actual_departure_delay_min` and `delayed` are recorded during or after the trip, so using them would be **target leakage**.

In [ ]:
# Colab ships pandas/sklearn/xgboost/lightgbm/shap; we add optuna and kagglehub.
%pip install -q optuna kagglehub shap tabulate

In [ ]:
"""Setup: imports, reproducibility, paths, plotting style."""
import json
import os
import random
import shutil
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Reproducibility --------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ---- Project paths (same layout as the GitHub repo) -------------------------
IN_COLAB = "google.colab" in sys.modules
PROJECT_ROOT = (Path("/content/transit-delay-prediction") if IN_COLAB
                else Path.cwd().parent)
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
MODEL_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"
for d in (RAW_DIR, PROCESSED_DIR, FIG_DIR, MODEL_DIR, SRC_DIR):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

KAGGLE_ID = "khushikyad001/public-transport-delays-with-weather-and-events"
DELAY_THRESHOLD_MIN = 5
TEST_SIZE = 0.20

# ---- Plot style: one accent hue, quiet grid, consistent across figures -----
BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110, "axes.edgecolor": "#d6d5d0", "grid.color": "#ebeae6",
    "axes.titleweight": "bold", "axes.titlesize": 12, "axes.labelcolor": MUTED,
    "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
})
DIVERGING = LinearSegmentedColormap.from_list("div", [BLUE, "#e9e8e4", "#e34948"])


def save_fig(fig, name: str) -> None:
    """Save a figure to reports/figures for the README."""
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"saved → {path.relative_to(PROJECT_ROOT)}")


print("Project root:", PROJECT_ROOT, "| Colab:", IN_COLAB)

---
# Phase 2 — Data understanding & cleaning

### 2.1 Load
The notebook first tries `kagglehub`, which works for public datasets without credentials. If that fails, it opens an upload box; download the CSV from the Kaggle page and select it.

**One parsing decision matters here:** by default pandas turns the string `"None"` into `NaN`. In this dataset `event_type = "None"` means *"no event"*, which is a valid category, not missing data. We read it as a literal string so we don't create a missing-data problem that isn't there.

In [ ]:
RAW_CSV = RAW_DIR / "public_transport_delays.csv"

if not RAW_CSV.exists():
    try:
        import kagglehub
        download_dir = Path(kagglehub.dataset_download(KAGGLE_ID))
        shutil.copy(next(download_dir.rglob("*.csv")), RAW_CSV)
        print("Downloaded via kagglehub")
    except Exception as exc:  # noqa: BLE001 - any failure → manual upload
        print(f"kagglehub failed ({exc}); please upload the CSV manually.")
        if not IN_COLAB:
            raise
        from google.colab import files
        uploaded = files.upload()
        RAW_CSV.write_bytes(next(iter(uploaded.values())))

# Only empty fields count as missing; the literal "None" stays a category.
df_raw = pd.read_csv(RAW_CSV, keep_default_na=False, na_values=[""])
naive_missing = pd.read_csv(RAW_CSV)["event_type"].isna().sum()

print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"event_type 'missing' with default parsing: {naive_missing:,} "
      f"→ with correct parsing: {df_raw['event_type'].isna().sum():,}")

In [ ]:
overview = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "n_missing": df_raw.isna().sum(),
    "pct_missing": (df_raw.isna().mean() * 100).round(2),
    "n_unique": df_raw.nunique(),
    "example": df_raw.iloc[0],
})
print(overview.to_string())
print(f"\nFully duplicated rows: {df_raw.duplicated().sum():,}")
print(f"Duplicated trip_id:    {df_raw['trip_id'].duplicated().sum():,}")

In [ ]:
df_raw.describe(include="number").T.round(2)

In [ ]:
# Category levels, so we know the real labels behind the feature flags.
for col in ["transport_type", "weather_condition", "event_type", "season"]:
    print(f"{col}: {df_raw[col].value_counts().to_dict()}")
print(f"route_id levels: {df_raw['route_id'].nunique()}, "
      f"origin_station levels: {df_raw['origin_station'].nunique()}")

### 2.2 Types, ordering, duplicates
- We combine `date` + `time` into a single `datetime` and **sort by it**. Every later split depends on time order.
- Duplicates are dropped. If the same trip appeared twice, one copy could land in train and the other in test, and the model would be graded on a trip it had already seen.

In [ ]:
df = df_raw.copy()
df["datetime"] = pd.to_datetime(
    df["date"].astype(str) + " " + df["time"].astype(str), errors="coerce")
df["date"] = pd.to_datetime(df["date"], errors="coerce")
print("Unparseable datetimes:", df["datetime"].isna().sum())

n_before = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset="trip_id", keep="first")
df = df.sort_values("datetime").reset_index(drop=True)
print(f"Removed {n_before - len(df)} duplicate rows → {len(df):,} rows")
print(f"Time span: {df['datetime'].min()} → {df['datetime'].max()} "
      f"({(df['datetime'].max() - df['datetime'].min()).days} days)")

### 2.3 Missing values & impossible values
**Policy:**
1. Rows with a **missing target** are dropped. We can't learn from them, and imputing a target invents labels.
2. Missing **features** are *not* filled here. They are imputed **inside the sklearn Pipeline** (median for numeric, `"Unknown"` for categorical), so the fill values are learned from the training fold only. Imputing here, before the split, would leak test-set statistics into training.
3. **Physically impossible** values (humidity > 100%, negative rainfall) are data errors. We set them to `NaN` so the pipeline imputes them.
4. **Statistical outliers in the target are kept.** A 40-minute delay is rare but real, and it's exactly what riders care about. We use MAE as the primary metric because it stays robust to those values without deleting them.

In [ ]:
VALID_RANGES = {
    "temperature_C": (-40, 50),
    "humidity_percent": (0, 100),
    "wind_speed_kmh": (0, 150),
    "precipitation_mm": (0, 200),
    "event_attendance_est": (0, 200_000),
    "traffic_congestion_index": (0, 100),
    "actual_departure_delay_min": (-60, 600),
    "actual_arrival_delay_min": (-60, 600),
}
TARGET = "actual_arrival_delay_min"

rows = []
for col, (lo, hi) in VALID_RANGES.items():
    bad = ~df[col].between(lo, hi) & df[col].notna()
    rows.append({"column": col, "valid_range": f"[{lo}, {hi}]",
                 "n_invalid": int(bad.sum()), "n_missing": int(df[col].isna().sum())})
    if col != TARGET:
        df.loc[bad, col] = np.nan
print(pd.DataFrame(rows).to_string(index=False))

n_before = len(df)
df = df[df[TARGET].notna() & df[TARGET].between(*VALID_RANGES[TARGET])]
print(f"\nDropped {n_before - len(df)} rows with missing/impossible target")

q1, q3 = df[TARGET].quantile([0.25, 0.75])
iqr_out = (df[TARGET] > q3 + 1.5 * (q3 - q1)).sum()
print(f"Target: mean={df[TARGET].mean():.2f}, median={df[TARGET].median():.1f}, "
      f"skew={df[TARGET].skew():.2f}, IQR outliers kept={iqr_out}")
print("Remaining missing values per column:\n",
      df.isna().sum()[lambda s: s > 0].to_string() or "none")

### 2.4 Synthetic-data audit
The Kaggle page describes this dataset as **synthetic**. Before modeling, we check how realistic its internal relationships are. Real data has to satisfy these checks, and generated data often fails them. The results set an upper bound on how much a model can learn.

In [ ]:
df["is_delayed"] = (df[TARGET] > DELAY_THRESHOLD_MIN).astype(int)
dep_hour = pd.to_numeric(df["scheduled_departure"].astype(str).str[:2], errors="coerce")
season_from_month = df["date"].dt.month.map({
    12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer", 9: "Autumn", 10: "Autumn", 11: "Autumn"})

audit = {
    "weekday column == weekday of date": (df["weekday"] == df["date"].dt.dayofweek).mean(),
    "season column == season of month (N. hemisphere)": (df["season"] == season_from_month).mean(),
    "'delayed' == arrival delay > 5": (df["delayed"] == df["is_delayed"]).mean(),
    "'delayed' == departure delay > 5":
        (df["delayed"] == (df["actual_departure_delay_min"] > 5)).mean(),
    "'Clear' weather but precipitation > 0":
        ((df["weather_condition"] == "Clear") & (df["precipitation_mm"] > 0)).mean(),
    "no event but attendance > 0":
        ((df["event_type"] == "None") & (df["event_attendance_est"] > 0)).mean(),
    "peak_hour flag == departure in 07-09h/16-19h":
        (df["peak_hour"] == dep_hour.isin(list(range(7, 10)) + list(range(16, 20)))).mean(),
}
audit_df = pd.DataFrame({"share_of_rows": audit}).round(3)
print(audit_df.to_string())

corr_dep_arr = df["actual_departure_delay_min"].corr(df[TARGET])
print(f"\nCorrelation departure delay ↔ arrival delay: {corr_dep_arr:.3f} "
      "(in real networks this is typically strongly positive)")
print(f"Delayed share (> {DELAY_THRESHOLD_MIN} min): {df['is_delayed'].mean():.1%}")

**How synthetic data limits the conclusions** (to be confirmed by the audit numbers above):
- **Relationships are whatever the generator encoded.** If columns were sampled independently, no model can find signal, and a "low" score is the *correct* result rather than a modeling failure.
- **Realistic dependencies can be missing** (departure delay → arrival delay, rain → precipitation, season ↔ month). Feature engineering based on domain knowledge only helps if the generator respected that domain.
- **The results don't transfer to a real network.** The pipeline, the validation design and the leakage controls do transfer. Those are what this project demonstrates.
- **Short time span** (a few weeks), so we can't learn seasonality or long-term trends.

In [ ]:
CLEAN_CSV = PROCESSED_DIR / "transit_clean.csv"
df.to_csv(CLEAN_CSV, index=False)
print(f"Saved cleaned data → {CLEAN_CSV.relative_to(PROJECT_ROOT)}  {df.shape}")

---
# Phase 3 — Exploratory data analysis
Every chart shows a **mean delay with 95% confidence intervals**. With only about 2k trips, some groups are small. If the intervals overlap, the difference might be noise, and we shouldn't claim it's a pattern.

In [ ]:
df["dep_hour"] = dep_hour
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
df["weekday_name"] = df["weekday"].map(dict(enumerate(WEEKDAYS)))
overall_mean = df[TARGET].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
sns.histplot(df[TARGET], bins=40, color=BLUE, edgecolor="white", ax=ax)
ax.axvline(DELAY_THRESHOLD_MIN, color=ORANGE, lw=2, ls="--")
ax.text(DELAY_THRESHOLD_MIN, ax.get_ylim()[1] * 0.95, f"  {DELAY_THRESHOLD_MIN}-min threshold",
        color=MUTED, va="top")
ax.set(title="Arrival delay distribution", xlabel="Arrival delay (min)", ylabel="Trips")
ax = axes[1]
sns.scatterplot(data=df, x="actual_departure_delay_min", y=TARGET, s=12, alpha=0.35,
                color=BLUE, edgecolor=None, ax=ax)
ax.set(title=f"Departure vs arrival delay (r = {corr_dep_arr:.2f})",
       xlabel="Departure delay (min) — leakage, EDA only", ylabel="Arrival delay (min)")
fig.tight_layout()
save_fig(fig, "01_delay_distribution")
plt.show()


def mean_delay_bar(col, ax, order=None, horizontal=False, title=None):
    """Mean arrival delay per category with 95% CI and the overall mean."""
    order = order or df.groupby(col)[TARGET].mean().sort_values(ascending=False).index.tolist()
    kw = dict(y=col, x=TARGET) if horizontal else dict(x=col, y=TARGET)
    sns.barplot(data=df, order=order, errorbar=("ci", 95), color=BLUE,
                err_kws={"color": MUTED, "linewidth": 1.2}, ax=ax, **kw)
    (ax.axvline if horizontal else ax.axhline)(overall_mean, color=ORANGE, lw=1.5, ls="--")
    ax.set_title(title or f"Mean arrival delay by {col}")
    ax.set_xlabel("Mean delay (min)" if horizontal else "")
    ax.set_ylabel("" if horizontal else "Mean delay (min)")


fig, ax = plt.subplots(figsize=(8, 0.28 * df["route_id"].nunique() + 1.5))
mean_delay_bar("route_id", ax, horizontal=True, title="Mean arrival delay by route (dashed = overall)")
save_fig(fig, "02_delay_by_route")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.pointplot(data=df, x="dep_hour", y=TARGET, errorbar=("ci", 95), color=BLUE,
              markersize=5, ax=axes[0])
axes[0].axhline(overall_mean, color=ORANGE, lw=1.5, ls="--")
axes[0].set(title="Mean arrival delay by departure hour", xlabel="Hour", ylabel="Mean delay (min)")
sns.pointplot(data=df, x="dep_hour", y="is_delayed", errorbar=("ci", 95), color=BLUE,
              markersize=5, ax=axes[1])
axes[1].axhline(df["is_delayed"].mean(), color=ORANGE, lw=1.5, ls="--")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[1].set(title=f"Share of trips delayed > {DELAY_THRESHOLD_MIN} min by hour",
            xlabel="Hour", ylabel="Delayed share")
fig.tight_layout()
save_fig(fig, "03_delay_by_hour")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
mean_delay_bar("weekday_name", axes[0], order=WEEKDAYS, title="By weekday")
mean_delay_bar("weather_condition", axes[1], title="By weather condition")
mean_delay_bar("event_type", axes[2], title="By event type")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
save_fig(fig, "04_delay_by_weekday_weather_event")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
mean_delay_bar("transport_type", axes[0], title="By transport type")
mean_delay_bar("holiday", axes[1], order=[0, 1], title="Holiday (1) vs not (0)")
mean_delay_bar("peak_hour", axes[2], order=[0, 1], title="Peak hour (1) vs not (0)")
fig.tight_layout()
save_fig(fig, "05_delay_by_mode_holiday_peak")
plt.show()

In [ ]:
# Continuous drivers: mean delay per decile, so non-linear effects stay visible.
NUM_DRIVERS = ["temperature_C", "precipitation_mm", "wind_speed_kmh",
               "traffic_congestion_index", "humidity_percent", "event_attendance_est"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey=True)
for ax, col in zip(axes.flat, NUM_DRIVERS):
    bins = pd.qcut(df[col], q=10, duplicates="drop")
    mids = df.groupby(bins, observed=True)[col].transform("median")
    sns.pointplot(x=mids.round(1), y=df[TARGET], errorbar=("ci", 95), color=BLUE,
                  markersize=4, ax=ax, native_scale=True)
    ax.axhline(overall_mean, color=ORANGE, lw=1.2, ls="--")
    ax.set(title=col, xlabel="Decile median", ylabel="Mean delay (min)")
fig.suptitle("Arrival delay across deciles of each continuous feature", fontweight="bold")
fig.tight_layout()
save_fig(fig, "06_delay_by_numeric_deciles")
plt.show()

In [ ]:
corr_cols = ["temperature_C", "humidity_percent", "wind_speed_kmh", "precipitation_mm",
             "event_attendance_est", "traffic_congestion_index", "holiday", "peak_hour",
             "weekday", "dep_hour", "actual_departure_delay_min", TARGET, "is_delayed"]
corr = df[corr_cols].corr(method="spearman")
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, cmap=DIVERGING, vmin=-1, vmax=1, center=0, annot=True,
            fmt=".2f", annot_kws={"size": 8}, linewidths=2, linecolor="white",
            cbar_kws={"shrink": 0.7, "label": "Spearman ρ"}, ax=ax)
ax.set_title("Spearman correlation (rank-based: robust to skew & outliers)")
fig.tight_layout()
save_fig(fig, "07_correlation_heatmap")
plt.show()

### 3.1 Statistical check behind the insights
Charts can mislead, so we test each driver. For categorical drivers we use **Kruskal-Wallis**, a rank-based test that doesn't assume delays are normally distributed. For numeric drivers we use **Spearman ρ**. With about 20 tests, a few will show p < 0.05 by chance alone, so we also apply a **Bonferroni** correction.

In [ ]:
from scipy import stats

cat_drivers = ["route_id", "transport_type", "weather_condition", "event_type", "season",
               "weekday_name", "dep_hour", "holiday", "peak_hour", "origin_station"]
num_drivers = NUM_DRIVERS
tests = []
for col in cat_drivers:
    groups = [g[TARGET].to_numpy() for _, g in df.groupby(col) if len(g) > 1]
    h, p = stats.kruskal(*groups)
    k, n = len(groups), len(df)
    tests.append({"feature": col, "test": "Kruskal-Wallis", "statistic": h,
                  "effect_size": max((h - k + 1) / (n - k), 0), "p_value": p})
for col in num_drivers:
    rho, p = stats.spearmanr(df[col], df[TARGET], nan_policy="omit")
    tests.append({"feature": col, "test": "Spearman", "statistic": rho,
                  "effect_size": abs(rho), "p_value": p})
tests_df = pd.DataFrame(tests)
tests_df["p_bonferroni"] = (tests_df["p_value"] * len(tests_df)).clip(upper=1)
tests_df["significant"] = tests_df["p_bonferroni"] < 0.05
print("effect_size = ε² for Kruskal-Wallis, |ρ| for Spearman\n")
print(tests_df.sort_values("p_value").round(4).to_string(index=False))

In [ ]:
# Fact sheet used to write the plain-English insights.
by_route = df.groupby("route_id")[TARGET].agg(["mean", "size"]).sort_values("mean")
facts = {
    "trips": len(df),
    "mean / median delay (min)": (round(overall_mean, 2), df[TARGET].median()),
    "share delayed > 5 min": f"{df['is_delayed'].mean():.1%}",
    "best / worst route": (by_route.index[0], round(by_route['mean'].iloc[0], 2),
                           by_route.index[-1], round(by_route['mean'].iloc[-1], 2)),
    "worst hour": int(df.groupby("dep_hour")[TARGET].mean().idxmax()),
    "delay by weather": df.groupby("weather_condition")[TARGET].mean().round(2).to_dict(),
    "delay by event": df.groupby("event_type")[TARGET].mean().round(2).to_dict(),
    "delay by mode": df.groupby("transport_type")[TARGET].mean().round(2).to_dict(),
    "peak vs off-peak": df.groupby("peak_hour")[TARGET].mean().round(2).to_dict(),
}
for k, v in facts.items():
    print(f"{k:28s} {v}")

### 3.2 Key insights
1. **Delays are frequent but spread suspiciously evenly.** Arrival delays run from −3 to +29 min with an almost flat distribution (mean ≈ 13.3 min), and about **75% of trips are more than 5 minutes late**. Real delays are right-skewed: most trips run close to on time, with a long tail of bad days. A flat shape is a typical sign of synthetic data. It also makes "delayed" the *majority* class, so predicting "delayed" for every trip already scores F1 ≈ 0.87.
2. **Departure delay says nothing about arrival delay (r = −0.01).** In a real network a trip that leaves 15 minutes late almost always arrives late, and this is usually the single strongest predictor. Here the two columns are independent.
3. **No route, hour or weekday stands out.** Route means range from 11.8 to 14.3 min and hourly means from 11.2 to 15.6 min, but every 95% confidence interval overlaps the overall mean. Rush hour (07–09h and 16–19h) is no worse than midday.
4. **Weather and events don't move delays.** Rain (13.6 min), snow (13.5), storm (13.3) and clear skies (13.5) are statistically the same, and so are protests (13.9) and festivals (12.7). On top of that, "Clear" days often record precipitation, which a real weather feed wouldn't do.
5. **Every pairwise correlation is close to zero (|ρ| ≤ 0.05)**, both between features and with the target. The columns look independently generated, so we should expect no model to beat the mean predictor, and Phase 5 tests exactly that.

---
# Phase 4 — Feature engineering
The feature code lives in **`src/features.py`**, not in the notebook, because the **Streamlit app must compute exactly the same features at prediction time**. Copy-pasting the code into two places is a classic way for training and serving to silently drift apart. The next cell writes the module, so the notebook stays self-contained on Colab.

| Group | Features | Why |
|---|---|---|
| Time | `dep_hour`, `hour_sin/cos`, `is_rush_hour`, `is_weekend`, `is_day_off`, `sched_duration_min` | Sine/cosine encoding makes 23:00 and 00:00 neighbours. Longer trips have more room to lose time |
| Weather | `is_rain`, `is_snow`, `is_extreme_temp`, `is_heavy_precip`, `is_high_wind`, `bad_weather_score` | Thresholded flags capture "it's bad enough to matter" effects that a linear term misses |
| Events | `has_event`, `log_event_attendance`, `is_large_event`, `event_in_rush` | Attendance is heavy-tailed, so we take the log. An event during rush hour should matter more than one off-peak |
| Interaction | `congestion_x_rush` | The same congestion level likely hurts more at peak times |
| Encoding | one-hot for low-cardinality columns; **TargetEncoder** for route/station | One-hot on ~100 route/station levels with 2k rows overfits. sklearn's TargetEncoder cross-fits, so a row's own label never leaks into its encoding |
| Scaling | StandardScaler on numerics | Needed for Ridge/logistic regression, harmless for trees, and keeps a single pipeline for all models |

**Event "proximity":** the dataset links an event to each trip but gives no coordinates or event times. So proximity here means *an event affecting this trip*, weighted by its size. With real data we'd compute distance to the venue and time to the event's start or end.

In [ ]:
%%writefile src/features.py
"""Feature engineering and preprocessing for transit delay prediction.

Design rule: every feature here must be computable *before the trip departs*
(timetable, weather forecast, scheduled events). The same functions are used
inside the training Pipeline and by the Streamlit app, so training-time and
serving-time features can never drift apart.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    StandardScaler,
    TargetEncoder,
)

# --------------------------------------------------------------------------
# Business / domain constants (documented here instead of magic numbers)
# --------------------------------------------------------------------------
DELAY_THRESHOLD_MIN = 5            # "delayed" = arrives more than 5 min late
RUSH_HOURS = set(range(7, 10)) | set(range(16, 20))   # 07-09h and 16-19h
WET_CONDITIONS = {"Rain", "Storm", "Drizzle", "Thunderstorm"}
SNOW_CONDITIONS = {"Snow", "Sleet", "Blizzard"}
EXTREME_COLD_C = 0.0
EXTREME_HEAT_C = 30.0
HEAVY_PRECIP_MM = 10.0
HIGH_WIND_KMH = 40.0
LARGE_EVENT_ATTENDANCE = 10_000
SEASON_BY_MONTH = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn",
}

# --------------------------------------------------------------------------
# Column contracts
# --------------------------------------------------------------------------
TARGET_REG = "actual_arrival_delay_min"
TARGET_CLF = "is_delayed"

# Known only during/after the trip -> must never be used as a feature.
LEAKAGE_COLUMNS = ["actual_departure_delay_min", "delayed", TARGET_REG]

# Raw columns the model pipeline expects as input (pre-departure info only).
RAW_INPUT_COLUMNS = [
    "transport_type", "route_id", "origin_station", "destination_station",
    "scheduled_departure", "scheduled_arrival",
    "weather_condition", "temperature_C", "humidity_percent",
    "wind_speed_kmh", "precipitation_mm",
    "event_type", "event_attendance_est",
    "traffic_congestion_index", "holiday", "peak_hour", "weekday", "season",
]

NUMERIC_FEATURES = [
    "temperature_C", "humidity_percent", "wind_speed_kmh", "precipitation_mm",
    "log_event_attendance", "traffic_congestion_index", "sched_duration_min",
    "dep_hour", "hour_sin", "hour_cos", "weekday", "bad_weather_score",
    "congestion_x_rush",
]
BINARY_FEATURES = [
    "is_rush_hour", "peak_hour", "is_weekend", "holiday", "is_day_off",
    "is_rain", "is_snow", "is_extreme_temp", "is_heavy_precip",
    "is_high_wind", "has_event", "is_large_event", "event_in_rush",
]
LOW_CARD_CATEGORICAL = [
    "transport_type", "weather_condition", "event_type", "season",
]
HIGH_CARD_CATEGORICAL = ["route_id", "origin_station", "destination_station"]


def clock_to_minutes(values: pd.Series) -> pd.Series:
    """Convert 'HH:MM' or 'HH:MM:SS' strings to minutes after midnight."""
    parts = values.astype(str).str.strip().str.split(":", expand=True)
    hours = pd.to_numeric(parts[0], errors="coerce")
    minutes = pd.to_numeric(parts[1], errors="coerce") if parts.shape[1] > 1 else 0
    return hours * 60 + minutes


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of ``df`` with engineered, leakage-free features added."""
    out = df.copy()

    # ---- Time features ----------------------------------------------------
    dep_min = clock_to_minutes(out["scheduled_departure"])
    arr_min = clock_to_minutes(out["scheduled_arrival"])
    out["dep_hour"] = (dep_min // 60).astype(float)
    # Cyclical encoding: 23:00 and 00:00 are neighbours, not 23 units apart.
    out["hour_sin"] = np.sin(2 * np.pi * dep_min / 1440)
    out["hour_cos"] = np.cos(2 * np.pi * dep_min / 1440)
    # Modulo handles trips that cross midnight.
    out["sched_duration_min"] = (arr_min - dep_min) % 1440
    out["is_rush_hour"] = out["dep_hour"].isin(RUSH_HOURS).astype(int)
    out["is_weekend"] = (out["weekday"] >= 5).astype(int)          # Mon=0
    out["is_day_off"] = ((out["is_weekend"] == 1) | (out["holiday"] == 1)).astype(int)

    # ---- Weather flags ----------------------------------------------------
    weather = out["weather_condition"].fillna("Unknown")
    out["is_rain"] = weather.isin(WET_CONDITIONS).astype(int)
    out["is_snow"] = weather.isin(SNOW_CONDITIONS).astype(int)
    out["is_extreme_temp"] = (
        (out["temperature_C"] < EXTREME_COLD_C) | (out["temperature_C"] > EXTREME_HEAT_C)
    ).astype(int)
    out["is_heavy_precip"] = (out["precipitation_mm"] > HEAVY_PRECIP_MM).astype(int)
    out["is_high_wind"] = (out["wind_speed_kmh"] > HIGH_WIND_KMH).astype(int)
    out["bad_weather_score"] = out[
        ["is_rain", "is_snow", "is_extreme_temp", "is_heavy_precip", "is_high_wind"]
    ].sum(axis=1)

    # ---- Event proximity --------------------------------------------------
    # The dataset links an event to each trip record (no coordinates), so
    # "proximity" = an event affecting this trip, scaled by its size.
    out["has_event"] = (out["event_type"].fillna("None") != "None").astype(int)
    attendance = out["event_attendance_est"].clip(lower=0)
    out["log_event_attendance"] = np.log1p(attendance)   # tames the heavy tail
    out["is_large_event"] = (attendance >= LARGE_EVENT_ATTENDANCE).astype(int)
    out["event_in_rush"] = out["has_event"] * out["is_rush_hour"]

    # ---- Interactions -----------------------------------------------------
    out["congestion_x_rush"] = out["traffic_congestion_index"] * out["is_rush_hour"]
    return out


def build_preprocessor(task: str, seed: int = 42) -> ColumnTransformer:
    """ColumnTransformer: impute + scale numerics, encode categoricals.

    All statistics (medians, encodings, scales) are learned inside the
    Pipeline from the training fold only, which prevents test-set leakage.
    """
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),   # needed for linear models; harmless for trees
    ])
    low_card = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    # Routes/stations have many levels: one-hot would create ~100 sparse
    # columns for 2k rows. TargetEncoder cross-fits internally, so a row's
    # own target never leaks into its encoding.
    high_card = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("target_enc", TargetEncoder(
            target_type="continuous" if task == "regression" else "binary",
            random_state=seed,
        )),
    ])
    binary = SimpleImputer(strategy="most_frequent")
    return ColumnTransformer(
        [
            ("num", numeric, NUMERIC_FEATURES),
            ("bin", binary, BINARY_FEATURES),
            ("cat", low_card, LOW_CARD_CATEGORICAL),
            ("hicard", high_card, HIGH_CARD_CATEGORICAL),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def make_pipeline(model, task: str, seed: int = 42) -> Pipeline:
    """Full raw-input -> prediction pipeline (features + preprocessing + model)."""
    return Pipeline([
        ("features", FunctionTransformer(add_features)),
        ("preprocess", build_preprocessor(task, seed)),
        ("model", model),
    ])

In [ ]:
import importlib

(SRC_DIR / "__init__.py").touch()
import src.features as F
importlib.reload(F)
from src.features import (HIGH_CARD_CATEGORICAL, LEAKAGE_COLUMNS, RAW_INPUT_COLUMNS,
                          TARGET_CLF, TARGET_REG, add_features, make_pipeline)

assert not set(LEAKAGE_COLUMNS) & set(RAW_INPUT_COLUMNS), "Leakage column in inputs!"

feat_preview = add_features(df[RAW_INPUT_COLUMNS])
new_cols = [c for c in feat_preview.columns if c not in RAW_INPUT_COLUMNS]
print(f"{len(new_cols)} engineered features:", new_cols)
feat_preview[new_cols].describe().T.round(2)

In [ ]:
# Sanity check: what does the preprocessor output look like?
_pre = make_pipeline(None, "regression", SEED)[:-1]
_out = _pre.fit_transform(df[RAW_INPUT_COLUMNS], df[TARGET_REG])
_names = _pre.named_steps["preprocess"].get_feature_names_out()
print(f"Model matrix: {_out.shape[0]} rows × {_out.shape[1]} columns")
print(list(_names))

---
# Phase 5 — Modeling

### 5.1 Time-aware split
A random split would let the model train on Tuesday's trips and be tested on Monday's. That leaks future information (weather spells, event weeks) and inflates scores. We use the following design instead:
- **Hold-out test set = the last 20% of the timeline.** It's touched **once**, at the very end.
- **Cross-validation = `TimeSeriesSplit` (5 folds) inside the training period.** Each fold trains on the past and validates on the period immediately after it, just as the model would be used in production.
- **Model selection uses CV scores, never test scores.** Picking the model with the best test score would turn the test set into a validation set, and the test score would stop being an honest estimate.

In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (average_precision_score, f1_score, mean_absolute_error,
                             mean_squared_error, precision_recall_curve, precision_score,
                             r2_score, recall_score, roc_auc_score)
from sklearn.model_selection import TimeSeriesSplit
from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBClassifier, XGBRegressor

data = df.sort_values("datetime").reset_index(drop=True)
split_idx = int(len(data) * (1 - TEST_SIZE))
train_df, test_df = data.iloc[:split_idx], data.iloc[split_idx:]
assert train_df["datetime"].max() <= test_df["datetime"].min(), "Split is not chronological!"

X_train, X_test = train_df[RAW_INPUT_COLUMNS], test_df[RAW_INPUT_COLUMNS]
y_train = {"regression": train_df[TARGET_REG], "classification": train_df[TARGET_CLF]}
y_test = {"regression": test_df[TARGET_REG], "classification": test_df[TARGET_CLF]}
tscv = TimeSeriesSplit(n_splits=5)

print(f"Train: {len(train_df):,} trips  {train_df['datetime'].min()} → {train_df['datetime'].max()}")
print(f"Test : {len(test_df):,} trips  {test_df['datetime'].min()} → {test_df['datetime'].max()}")
print(f"Delayed share — train {y_train['classification'].mean():.1%}, "
      f"test {y_test['classification'].mean():.1%}")

### 5.2 Model zoo & evaluation helpers
- **Baselines:** the mean predictor and class prior show the "no-skill" floor. Ridge and logistic regression show what a simple linear model achieves.
- **Random Forest:** bagging, robust, few hyperparameters to get wrong.
- **XGBoost / LightGBM:** gradient boosting, usually the strongest choice on tabular data.
- **Class imbalance:** instead of reweighting classes, we **tune the decision threshold** on out-of-fold predictions. This keeps the probabilities interpretable and lets the business choose its trade-off between precision and recall.

In [ ]:
TREE_MODELS = ["Random Forest", "XGBoost", "LightGBM"]


def build_model(name: str, task: str, params: dict | None = None):
    """Return an unfitted estimator with sensible, seeded defaults."""
    reg = task == "regression"
    params = params or {}
    if name == "Baseline":
        return DummyRegressor(strategy="mean") if reg else DummyClassifier(strategy="prior")
    if name == "Linear":
        return Ridge(alpha=1.0) if reg else LogisticRegression(max_iter=2000, random_state=SEED)
    if name == "Random Forest":
        cls = RandomForestRegressor if reg else RandomForestClassifier
        base = dict(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
    elif name == "XGBoost":
        cls = XGBRegressor if reg else XGBClassifier
        base = dict(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8,
                    colsample_bytree=0.8, n_jobs=-1, random_state=SEED)
        if not reg:
            base["eval_metric"] = "logloss"
    elif name == "LightGBM":
        cls = LGBMRegressor if reg else LGBMClassifier
        base = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, subsample=0.8,
                    subsample_freq=1, colsample_bytree=0.8, n_jobs=-1,
                    random_state=SEED, verbose=-1)
    else:
        raise ValueError(name)
    base.update(params)
    return cls(**base)


def time_cv(pipe, X, y, task):
    """TimeSeriesSplit CV → per-fold metrics + out-of-fold predictions."""
    folds, oof = [], np.full(len(y), np.nan)
    for tr, va in tscv.split(X):
        model = clone(pipe).fit(X.iloc[tr], y.iloc[tr])
        if task == "regression":
            pred = model.predict(X.iloc[va])
            folds.append({"MAE": mean_absolute_error(y.iloc[va], pred),
                          "RMSE": np.sqrt(mean_squared_error(y.iloc[va], pred)),
                          "R2": r2_score(y.iloc[va], pred)})
        else:
            pred = model.predict_proba(X.iloc[va])[:, 1]
            folds.append({"ROC_AUC": roc_auc_score(y.iloc[va], pred),
                          "PR_AUC": average_precision_score(y.iloc[va], pred)})
        oof[va] = pred
    return pd.DataFrame(folds), oof


def best_f1_threshold(y_true, proba) -> float:
    """Threshold that maximises F1 on out-of-fold (training-period) predictions."""
    mask = ~np.isnan(proba)
    prec, rec, thr = precision_recall_curve(np.asarray(y_true)[mask], proba[mask])
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    return float(thr[np.argmax(f1)])


def test_metrics(pipe, task, threshold=0.5) -> dict:
    """Score a fitted pipeline once on the hold-out test period."""
    yt = y_test[task]
    if task == "regression":
        pred = pipe.predict(X_test)
        return {"test_MAE": mean_absolute_error(yt, pred),
                "test_RMSE": np.sqrt(mean_squared_error(yt, pred)),
                "test_R2": r2_score(yt, pred)}
    proba = pipe.predict_proba(X_test)[:, 1]
    label = (proba >= threshold).astype(int)
    return {"test_ROC_AUC": roc_auc_score(yt, proba),
            "test_PR_AUC": average_precision_score(yt, proba),
            "threshold": threshold,
            "test_F1": f1_score(yt, label, zero_division=0),
            "test_precision": precision_score(yt, label, zero_division=0),
            "test_recall": recall_score(yt, label, zero_division=0)}


RESULTS = {"regression": [], "classification": []}
FITTED = {"regression": {}, "classification": {}}


def run_model(name, task, params=None, label=None):
    """CV on train → refit on all train → single test evaluation → log row."""
    label = label or name
    pipe = make_pipeline(build_model(name, task, params), task, SEED)
    t0 = time.time()
    folds, oof = time_cv(pipe, X_train, y_train[task], task)
    threshold = 0.5
    if task == "classification" and name != "Baseline":
        threshold = best_f1_threshold(y_train[task], oof)
    pipe.fit(X_train, y_train[task])
    row = {"model": label}
    for metric in folds.columns:
        row[f"cv_{metric}"] = folds[metric].mean()
        row[f"cv_{metric}_std"] = folds[metric].std()
    row.update(test_metrics(pipe, task, threshold))
    row["fit_seconds"] = time.time() - t0
    RESULTS[task].append(row)
    FITTED[task][label] = pipe
    return row

### 5.3 Baselines + default models (both tasks)

In [ ]:
for task in ("regression", "classification"):
    for name in ["Baseline", "Linear"] + TREE_MODELS:
        r = run_model(name, task)
        key = "cv_MAE" if task == "regression" else "cv_ROC_AUC"
        print(f"[{task:14s}] {name:14s} {key}={r[key]:.3f}  ({r['fit_seconds']:.1f}s)")

### 5.4 Hyperparameter tuning with Optuna
**Why Optuna over grid or random search:** its TPE sampler learns from earlier trials and spends the budget on promising regions, so it usually finds good settings in far fewer trials. The objective is the **mean `TimeSeriesSplit` CV score**, so tuning never sees the test set. The sampler is seeded for reproducibility.

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TRIALS = 30           # raise for a better search; ~10 min total on Colab CPU
TIMEOUT_SEC = 240       # hard cap per study


def suggest_params(trial, name):
    if name == "Random Forest":
        return {"n_estimators": trial.suggest_int("n_estimators", 100, 400, step=100),
                "max_depth": trial.suggest_int("max_depth", 3, 14),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 40),
                "max_features": trial.suggest_float("max_features", 0.2, 1.0)}
    common = {"n_estimators": trial.suggest_int("n_estimators", 100, 600, step=50),
              "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
              "subsample": trial.suggest_float("subsample", 0.5, 1.0),
              "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
              "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True)}
    if name == "XGBoost":
        common.update(max_depth=trial.suggest_int("max_depth", 2, 6),
                      min_child_weight=trial.suggest_float("min_child_weight", 1, 30, log=True))
    else:  # LightGBM
        common.update(num_leaves=trial.suggest_int("num_leaves", 4, 63),
                      min_child_samples=trial.suggest_int("min_child_samples", 5, 100))
    return common


BEST_PARAMS = {"regression": {}, "classification": {}}
for task in ("regression", "classification"):
    metric, direction = (("MAE", "minimize") if task == "regression"
                         else ("ROC_AUC", "maximize"))
    for name in TREE_MODELS:
        def objective(trial, name=name, task=task, metric=metric):
            pipe = make_pipeline(build_model(name, task, suggest_params(trial, name)), task, SEED)
            folds, _ = time_cv(pipe, X_train, y_train[task], task)
            return folds[metric].mean()

        study = optuna.create_study(direction=direction,
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC)
        BEST_PARAMS[task][name] = study.best_params
        r = run_model(name, task, study.best_params, label=f"{name} (tuned)")
        print(f"[{task:14s}] {name:14s} best cv_{metric}={study.best_value:.3f} "
              f"after {len(study.trials)} trials")

### 5.5 Comparison tables
The model is selected by **CV score** (the primary metric). Test columns are reported for transparency; they are not used for selection.

In [ ]:
def results_table(task):
    res = pd.DataFrame(RESULTS[task]).set_index("model")
    if task == "regression":
        res = res.sort_values("cv_MAE")
        cols = ["cv_MAE", "cv_MAE_std", "cv_RMSE", "cv_R2", "test_MAE", "test_RMSE", "test_R2"]
    else:
        res = res.sort_values("cv_ROC_AUC", ascending=False)
        cols = ["cv_ROC_AUC", "cv_ROC_AUC_std", "cv_PR_AUC", "test_ROC_AUC", "test_PR_AUC",
                "threshold", "test_F1", "test_precision", "test_recall"]
    return res[cols].round(3)


reg_table, clf_table = results_table("regression"), results_table("classification")
print("REGRESSION (sorted by CV MAE, lower is better)")
print(reg_table.to_string())
print("\nCLASSIFICATION (sorted by CV ROC-AUC, higher is better)")
print(clf_table.to_string())
print(f"\nTest-period delayed share (PR-AUC of a random model): "
      f"{y_test['classification'].mean():.3f}")

reg_table.to_markdown(PROJECT_ROOT / "reports" / "results_regression.md")
clf_table.to_markdown(PROJECT_ROOT / "reports" / "results_classification.md")

BEST = {"regression": reg_table.index[0], "classification": clf_table.index[0]}
BEST_TREE = {t: next(m for m in tbl.index if any(m.startswith(n) for n in TREE_MODELS))
             for t, tbl in (("regression", reg_table), ("classification", clf_table))}
print("\nSelected (by CV):", BEST, "\nBest tree model (for SHAP):", BEST_TREE)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, (task, tbl, metric, xlabel) in zip(axes, [
        ("regression", reg_table, "cv_MAE", "CV MAE (min) — lower is better"),
        ("classification", clf_table, "cv_ROC_AUC", "CV ROC-AUC — higher is better")]):
    order = tbl.sort_values(metric, ascending=(task == "classification")).index
    ax.barh(order, tbl.loc[order, metric], xerr=tbl.loc[order, f"{metric}_std"],
            color=[ORANGE if m == BEST[task] else BLUE for m in order],
            error_kw={"ecolor": MUTED, "lw": 1}, height=0.6)
    lo = tbl[metric].min() - 2 * tbl[f"{metric}_std"].max()
    ax.set_xlim(left=max(lo, 0))
    ax.set(title=f"{task.title()}: model comparison (±1 sd across folds)", xlabel=xlabel)
fig.tight_layout()
save_fig(fig, "08_model_comparison")
plt.show()

### 5.6 Is there real signal? Permutation test
This is the most important check for a synthetic dataset. We shuffle the target many times, refit, and see how often a model trained on **pure noise** scores as well as the real model. If p ≥ 0.05, the model's apparent skill is indistinguishable from chance. That isn't a bug: it's the finding, and it's a strong interview story.

In [ ]:
from sklearn.model_selection import permutation_test_score

PERM_RESULTS = {}
for task, scoring in (("regression", "neg_mean_absolute_error"), ("classification", "roc_auc")):
    name = BEST_TREE[task]
    base = name.replace(" (tuned)", "")
    params = BEST_PARAMS[task].get(base) if "(tuned)" in name else None
    pipe = make_pipeline(build_model(base, task, params), task, SEED)
    score, perm_scores, p = permutation_test_score(
        pipe, X_train, y_train[task], cv=tscv, scoring=scoring,
        n_permutations=30, random_state=SEED, n_jobs=1)
    PERM_RESULTS[task] = {"model": name, "score": score, "perm_mean": perm_scores.mean(),
                          "perm_std": perm_scores.std(), "p_value": p}
    print(f"{task:14s} {name:22s} real={score:.3f}  shuffled={perm_scores.mean():.3f}"
          f"±{perm_scores.std():.3f}  p={p:.3f}")

---
# Phase 6 — Explainability
We explain the **best tree model** with three complementary views:
1. **Built-in importance:** fast, but biased toward high-cardinality and continuous features.
2. **SHAP:** consistent attribution for each prediction, plus the *direction* of each effect (does high congestion push delay up or down?).
3. **Permutation importance on the test period:** model-agnostic. If shuffling a feature doesn't hurt test performance, the model doesn't really rely on it, whatever SHAP says.

In [ ]:
import shap
from sklearn.inspection import permutation_importance


def transformed_matrix(pipe, X):
    """Run features + preprocessing and return a named DataFrame."""
    feats = pipe.named_steps["features"].transform(X)
    mat = pipe.named_steps["preprocess"].transform(feats)
    names = pipe.named_steps["preprocess"].get_feature_names_out()
    return pd.DataFrame(np.asarray(mat, dtype=float), columns=names, index=X.index)


SHAP_SUMMARY = {}
for task in ("regression", "classification"):
    name = BEST_TREE[task]
    pipe = FITTED[task][name]
    model = pipe.named_steps["model"]
    Xt = transformed_matrix(pipe, X_test)

    # 1) Built-in importance
    imp = pd.Series(model.feature_importances_, index=Xt.columns).nlargest(15)[::-1]
    # 2) SHAP
    try:
        sv = shap.TreeExplainer(model).shap_values(Xt)
    except Exception as exc:  # noqa: BLE001 - version mismatch → model-agnostic SHAP
        print(f"TreeExplainer failed ({exc}); using model-agnostic SHAP")
        f = (model.predict if task == "regression"
             else (lambda a: model.predict_proba(a)[:, 1]))
        sv = shap.Explainer(f, shap.sample(Xt, 100, random_state=SEED))(Xt).values
    if isinstance(sv, list):          # older SHAP: one array per class
        sv = sv[1]
    sv = np.asarray(sv)
    if sv.ndim == 3:                  # newer SHAP: (rows, features, classes)
        sv = sv[:, :, 1]
    mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=Xt.columns)
    direction = pd.Series({c: np.corrcoef(Xt[c], sv[:, i])[0, 1] if Xt[c].std() > 0 else 0.0
                           for i, c in enumerate(Xt.columns)})
    SHAP_SUMMARY[task] = pd.DataFrame({"mean_abs_shap": mean_abs,
                                       "direction (+ = higher value → more delay)": direction}
                                      ).sort_values("mean_abs_shap", ascending=False)

    fig, ax = plt.subplots(figsize=(7, 5.5))
    ax.barh(imp.index, imp.values, color=BLUE, height=0.6)
    ax.set(title=f"{task.title()} — {name}: built-in importance", xlabel="Importance")
    fig.tight_layout()
    save_fig(fig, f"09_{task}_builtin_importance")
    plt.show()

    plt.figure()
    shap.summary_plot(sv, Xt, max_display=15, show=False)
    plt.title(f"{task.title()} — SHAP summary ({name})", fontweight="bold")
    save_fig(plt.gcf(), f"10_{task}_shap_summary")
    plt.show()

    print(f"\n{task.upper()} — top 10 features by mean |SHAP| ({name})")
    print(SHAP_SUMMARY[task].head(10).round(3).to_string())

In [ ]:
# 3) Permutation importance on RAW input columns (what a user can actually change).
PERM_IMP = {}
for task, scoring in (("regression", "neg_mean_absolute_error"), ("classification", "roc_auc")):
    pipe = FITTED[task][BEST_TREE[task]]
    pi = permutation_importance(pipe, X_test, y_test[task], scoring=scoring,
                                n_repeats=10, random_state=SEED, n_jobs=1)
    PERM_IMP[task] = pd.DataFrame({"mean": pi.importances_mean, "std": pi.importances_std},
                                  index=X_test.columns).sort_values("mean", ascending=False)
    top = PERM_IMP[task].head(12)[::-1]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh(top.index, top["mean"], xerr=top["std"], color=BLUE, height=0.6,
            error_kw={"ecolor": MUTED, "lw": 1})
    ax.axvline(0, color=MUTED, lw=1)
    ax.set(title=f"{task.title()}: permutation importance on test period",
           xlabel=f"Drop in {scoring} when shuffled (±1 sd)")
    fig.tight_layout()
    save_fig(fig, f"11_{task}_permutation_importance")
    plt.show()
    print(PERM_IMP[task].head(10).round(4).to_string())

### 6.1 What actually causes delays?
**In this dataset, nothing we can measure.** The evidence comes from three independent angles:
- **Performance:** the best tuned model reaches CV MAE 7.95 min against 7.97 for the mean predictor, and on the test period the mean predictor actually wins (8.16 vs 8.17). The best classifier has a test ROC-AUC of **0.46**, worse than a coin flip.
- **Permutation test:** models trained on *shuffled* targets score just as well (p = 0.65 for regression, p = 0.26 for classification). The apparent skill is chance.
- **Importance:** SHAP ranks `origin_station` and `route_id` highest. That isn't evidence of a cause: target-encoded, high-cardinality features give a tree the most ways to fit noise. Permutation importance on the test period confirms it. Shuffling the most important input raises MAE by just **0.03 min (0.4%)**.

**Lesson:** SHAP explains *the model*, not *the world*. A feature can top a SHAP plot while carrying zero real signal, which is why we check importance against held-out performance and a permutation test.

**What would drive delays in a real network, and how this pipeline would capture them:** knock-on delay from earlier trips (lag features from real-time feeds such as GTFS-Realtime), traffic congestion at peak times (`congestion_x_rush`), severe weather interacting with mode (buses suffer, metros don't), and large events near the route ending at a known time.

---
# Phase 7 — Export artefacts for the Streamlit app
We save the **final pipelines** (features + preprocessing + model in a single object, so the app can't apply different preprocessing), plus a `metadata.json` holding the dropdown options, input ranges, the tuned threshold and the metrics. **Versions are pinned** in `requirements.txt`, because a pickled sklearn model only loads reliably with the same library versions.

In [ ]:
import joblib
from importlib.metadata import version

final = {task: FITTED[task][BEST[task]] for task in ("regression", "classification")}
clf_row = clf_table.loc[BEST["classification"]]
reg_row = reg_table.loc[BEST["regression"]]

joblib.dump(final["regression"], MODEL_DIR / "delay_regressor.joblib", compress=3)
joblib.dump(final["classification"], MODEL_DIR / "delay_classifier.joblib", compress=3)

cat_cols = ["transport_type", "route_id", "origin_station", "destination_station",
            "weather_condition", "event_type"]
num_cols = ["temperature_C", "humidity_percent", "wind_speed_kmh", "precipitation_mm",
            "event_attendance_est", "traffic_congestion_index"]
duration = add_features(train_df[RAW_INPUT_COLUMNS])["sched_duration_min"]
metadata = {
    "raw_input_columns": RAW_INPUT_COLUMNS,
    "categories": {c: sorted(train_df[c].dropna().astype(str).unique().tolist()) for c in cat_cols},
    "numeric_ranges": {c: {"min": float(train_df[c].min()), "max": float(train_df[c].max()),
                           "median": float(train_df[c].median())} for c in num_cols},
    "duration_range": {"min": int(duration.min()), "max": int(duration.max()),
                       "median": int(duration.median())},
    "delay_threshold_min": DELAY_THRESHOLD_MIN,
    "classification_threshold": float(clf_row["threshold"]),
    "models": {"regression": BEST["regression"], "classification": BEST["classification"]},
    "metrics": {"regression": reg_row.to_dict(), "classification": clf_row.to_dict()},
    "permutation_test": {t: {k: (float(v) if not isinstance(v, str) else v)
                             for k, v in r.items()} for t, r in PERM_RESULTS.items()},
    "train_period": [str(train_df["datetime"].min()), str(train_df["datetime"].max())],
    "seed": SEED,
}
(MODEL_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, default=float))

PINNED = ["pandas", "numpy", "scikit-learn", "xgboost", "lightgbm", "shap", "optuna",
          "matplotlib", "seaborn", "scipy", "joblib"]
reqs = [f"{p}=={version(p)}" for p in PINNED] + ["streamlit>=1.35", "tabulate"]
(PROJECT_ROOT / "requirements.txt").write_text("\n".join(reqs) + "\n")
print("Python", sys.version.split()[0])
print("\n".join(reqs))
for f in sorted(MODEL_DIR.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:6.2f} MB")

In [ ]:
# Smoke test: one raw trip through the saved pipelines, exactly as the app does it.
reg_loaded = joblib.load(MODEL_DIR / "delay_regressor.joblib")
clf_loaded = joblib.load(MODEL_DIR / "delay_classifier.joblib")
sample = X_test.iloc[[0]]
print(sample.T)
print(f"\nPredicted delay: {reg_loaded.predict(sample)[0]:.1f} min | "
      f"P(delay > {DELAY_THRESHOLD_MIN} min) = {clf_loaded.predict_proba(sample)[0, 1]:.1%} | "
      f"actual: {test_df[TARGET_REG].iloc[0]} min")

In [ ]:
# Final summary to paste back into the chat.
print("=" * 70)
print("REGRESSION\n", reg_table.to_string())
print("\nCLASSIFICATION\n", clf_table.to_string())
print("\nPERMUTATION TEST\n", pd.DataFrame(PERM_RESULTS).T.to_string())
for task in ("regression", "classification"):
    print(f"\nTOP SHAP ({task})\n", SHAP_SUMMARY[task].head(8).round(3).to_string())
print("=" * 70)

In [ ]:
# Bundle everything for GitHub (raw data excluded: Kaggle licence).
import zipfile

bundle = (Path("/content/transit_delay_artifacts.zip") if IN_COLAB
          else PROJECT_ROOT / "transit_delay_artifacts.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in PROJECT_ROOT.rglob("*"):
        rel = path.relative_to(PROJECT_ROOT)
        if (path.is_file() and rel.parts[0] != "data" and path != bundle
                and "__pycache__" not in rel.parts):
            zf.write(path, rel)
print(f"Bundle: {bundle} ({bundle.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(bundle))